This lesson is about a new type of task in NLP called Named Entity Recognition (NER). So far we used to have input and only needed to predict one vector (context vector for sentiment classification). Working with NER is a bit different. The aim is to classify 'entities' into predefined classes such as Person, Date, Location, etc. In this example we work with Person, Organization, Location, Miscelaneous.

In [ ]:
from datasets import load_dataset

In [ ]:
ds = load_dataset("lhoestq/conll2003")

In [ ]:
ds['train'][0] # Lets see what the first item in the dataset looks like

We are interested in 'ner_tags'. Here are the class numbers and the labels.

{'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8}

The B indicates that the token is the first component of the entity. For example Barack in Barack Obama would be B-PER, while Obama would be I-PER, since its the continuation of Barack.

In [ ]:
from transformers import BertTokenizerFast

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained('bert-base-cased') # load the tokenizer

In [ ]:
ds['train'][0]

As usual with the pipeline, we need to preprocess the data before we feed it into the model.

In [ ]:
def preprocess(example):
  """
  Function to preprocess items from our datasets
  Input: example containing tokens, ner_tags and some other stuff
  Output: ner tags ('labels') and some other stuff
  """
  tokenized = tokenizer(example['tokens'], is_split_into_words=True, truncation=True) # is_split_into_words tells our tokenizer that it receives input split into words
  word_ids = tokenized.word_ids() # Get word ids starting from 0, subsequent tokens of the same word get the same id

  aligned_labels = []
  previous_word_id = None

  for word_id in word_ids:
      if word_id is None: # Special tokens like [CLS] get None as a word_id. To make our model ignore them we set its label to -100
          aligned_labels.append(-100)
      elif word_id == previous_word_id: # Set to -100 if it's a continuation of the previous token (for example la, ##mb -> ##mb gets -100)
          aligned_labels.append(-100)
      else:
          aligned_labels.append(example['ner_tags'][word_id]) # in other cases set label to ner class
      previous_word_id = word_id

  tokenized['labels'] = aligned_labels
  return tokenized

In [ ]:
# Map the preprocess function to train and val sets
train_ds = ds['train'].map(preprocess)
val_ds = ds['validation'].map(preprocess)

In [ ]:
from transformers import BertForTokenClassification

In [ ]:
model = BertForTokenClassification.from_pretrained('bert-base-cased', num_labels=9) # Load the model with 9 labels matching to our ner tags

In [ ]:
# Setup collator, training arguments and the trainer
from transformers import DataCollatorForTokenClassification, TrainingArguments, Trainer

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir='./ner-results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    eval_strategy='epoch'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator
)

In [ ]:
# Train the model
trainer.train()

In [ ]:
# Install a library for model evaluation
!pip install seqeval

In [ ]:
predictions = trainer.predict(val_ds) # Make predictions (in form of probabilities)
logits = predictions.predictions # Select all the probabilities of all labels for all data points
print(logits.shape) # 3250 data points, 151 classifications per point (padding added to reach 151), 9 probabilities for classes 0-8
labels = predictions.label_ids # same thing without the probabilities
print(labels.shape)

In [ ]:
print(logits[6][1]) # Get the predicted probabilities for the 2nd word in the 7th review (Class 3 is the highest probability)
print(labels[6]) # Get the true class for that word (Class 3)

In [ ]:
import numpy as np
pred_ids = np.argmax(logits, axis=-1)  # pick the highest scoring probability (label) for every token

After argmax, we choose the highest probability so the shape goes (3250, 151, 9) -> (3250, 151). And now since the shape of the predictions and the true values is the same, we're ready to compare them.

In [ ]:
test_ds = ds['test'].map(preprocess)
test_predictions = trainer.predict(test_ds)

In [ ]:
pred_ids_test = np.argmax(test_predictions.predictions, axis=-1)

In [ ]:
from seqeval.metrics import classification_report

label_names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

true_labels = []
pred_labels = []

for i in range(len(labels)): # loop through each example
    true_seq = [] # collect the true labels here
    pred_seq = [] # collect the model predicted labels here
    for j in range(len(labels[i])): # loop through each token of that example
        if labels[i][j] != -100:  # skip padding and subword tokens
            true_seq.append(label_names[labels[i][j]])
            pred_seq.append(label_names[pred_ids[i][j]])
    true_labels.append(true_seq)
    pred_labels.append(pred_seq)

print(classification_report(true_labels, pred_labels))

In [ ]:
# Do the same for the test set

true_labels_test = []
pred_labels_test = []

for i in range(len(test_predictions.label_ids)):
    true_seq_test = []
    pred_seq_test = []
    for j in range(len(test_predictions.label_ids[i])):
        if test_predictions.label_ids[i][j] != -100:
            true_seq_test.append(label_names[test_predictions.label_ids[i][j]])
            pred_seq_test.append(label_names[pred_ids_test[i][j]])
    true_labels_test.append(true_seq_test)
    pred_labels_test.append(pred_seq_test)

print(classification_report(true_labels_test, pred_labels_test))

As we can see the model got a micro avg f1-score of 91%, which is quite impressive. it still kinda struggles to predict the misc class accurately, since a lot of different words are considered misc, and this class is vague.